In [ ]:
import json
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_cohere import ChatCohere
from langchain_core.output_parsers import StrOutputParser
import pandas as pd
import time

In [ ]:
with open("data/gods.json", "r") as file:
    gods = json.load(file)
    
print('Total number of items:', len(gods))

In [ ]:
gods[0]

In [ ]:
system_message = (
    "تو یک دستیار هستی که قراره با توجه به اطلاعات داخل {information} به سوالاتی که ازت میپرسم  توی 4 الی 5 کلمه پاسخ بدی و اگر چنانچه جواب سوالی رو بلد نبود نیاز نیست"
    "از خودت جواب در بیاری و صرفا بنویس نمیدانم"
)

In [ ]:
examples = [
    {"question": " خواهر زئوس کیست؟" , "answer": "هستیا"},
    {"question": "چه کسی فرد سالخورده و نشسته روی الاغ است؟" , "answer": "هفایستوس"} , 
    {"question": "چه کسی یک جنگجو متولد شده است؟" , "answer": "آتنا"} , 
    {"question": "آتنا چه چیز هایی همراه خود دارد؟" , "answer": "سپر و شمشیر و جغد"} , 
    {"question": "آرتمیس به چه چیز هایی مجهز است؟" , "answer": "تیر و کمان"} , 
    {"question": "ویژگی های ظاهری دیونیسوس رو بیان کن." , "answer": "موی بلند و بی رحم و ریش بلند"} ,
    {"question": " حیوانات هرمس چه هستند؟" , "answer": "لاک پشت و خرگوش"} , 
    {"question": " پوسوئیدون چه چیز هایی با خود دارد؟" , "answer": "نیزه سه شاخ"},
    {"question": "همسر دروغین هفایستوس چه کسی است؟" , "answer": "آفرودیت"},
    {"question": "چه کسی ورزشکار است و ریش بلند دارد؟" , "answer": "هرمس"},
]

In [ ]:
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}")
])

# ۲. ساخت پرامپت چند نمونه‌ای
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt, 
    examples=examples
)

# تبدیل اطلاعات خدایان به یک رشته متنی تمیز
gods_text = ""
for g in gods:
    gods_text += f"نام: {g.get('name')}\nتوضیحات: {g.get('description')}\nظاهر: {g.get('appearance')}\nویژگی‌ها: {g.get('features')}\n\n"


formatted_system_message = system_message.format(information=gods_text)

prompt = ChatPromptTemplate.from_messages([
    ("system", "تو اطلاعات خوبی درباره خدایان داری"), 
    ("system", formatted_system_message), # متن نهایی شده سیستم بدون متغیر آزاد
    few_shot_prompt, 
    ("human", "{question}")
])

In [ ]:
model = ChatCohere(
    model="command-r-08-2024", # حتماً مطمئن شوید دقیقاً همین عبارت است
    temperature=0.0,
    cohere_api_key="COHERE_API_KEY"
)

In [ ]:
parser = StrOutputParser()
chain = prompt | model | parser

In [ ]:
questions = pd.read_csv("data/questions.csv")
questions

In [ ]:
answers = []

# حلقه زدن روی ردیف‌های دیتافریم سوالات
for idx, row in questions.iterrows():
    q = row['question']
    
    # ارسال سوال به مدل در قالب دیکشنری
    res = chain.invoke({"question": q})
    res_text = res if isinstance(res, str) else res.content
    answers.append(res_text)

    if (idx + 1) % 15 == 0:
        print(f"به {idx + 1} سوال پاسخ داده شد. ۶۵ ثانیه توقف برای جلوگیری از خطا...")
        time.sleep(65)

submission = pd.DataFrame({"answer": answers})
submission